In [1]:
import pandas as pd

cms = pd.read_csv("../data_raw/Hospital_General_Information.csv")
cms.shape

(5426, 38)

In [2]:
cols = [
    "Facility ID",
    "Facility Name",
    "City/Town",
    "County/Parish",
    "State",
    "Hospital Type",
    "Hospital Ownership",
    "Emergency Services",
    "Hospital overall rating",
] #Columns that we are interested in

In [3]:
pa = cms[cms["State"] == "PA"][cols].copy()
pa.shape #(PA Hospitals, columns)

(188, 9)

In [4]:
#Tells us how many 1,2,3,4,5 star hospitals are in PA

pa["Hospital overall rating"].value_counts().sort_index()

Hospital overall rating
1                 4
2                25
3                41
4                37
5                15
Not Available    66
Name: count, dtype: int64

In [5]:
#Average rating by county, convert text to number

pa["Hospital overall rating"] = pd.to_numeric(pa["Hospital overall rating"], errors="coerce")
pa = pa.dropna(subset=["Hospital overall rating"])

pa.groupby("County/Parish")["Hospital overall rating"].mean().sort_values(ascending=False).head(10)

County/Parish
CENTRE         5.000000
LEBANON        5.000000
UNION          5.000000
MIFFLIN        5.000000
NORTHAMPTON    4.333333
ADAMS          4.000000
LEHIGH         4.000000
DAUPHIN        4.000000
COLUMBIA       4.000000
MONTOUR        4.000000
Name: Hospital overall rating, dtype: float64

In [6]:
#Counting how many hospitals are in each county

county_summary = (
    pa.groupby("County/Parish")
      .agg(
          avg_rating=("Hospital overall rating", "mean"),
          hospital_count=("Facility ID", "count")
      )
      .sort_values("avg_rating", ascending=False)
)

county_summary.head(10)

,avg_rating,hospital_count
County/Parish,,
CENTRE,5.000000,1
LEBANON,5.000000,2
UNION,5.000000,1
MIFFLIN,5.000000,1
NORTHAMPTON,4.333333,3
ADAMS,4.000000,1
LEHIGH,4.000000,1
DAUPHIN,4.000000,2
COLUMBIA,4.000000,1


In [7]:
#Filtering for counties with at least 4 hospitals

county_summary[county_summary["hospital_count"] >= 4] \
    .sort_values("avg_rating", ascending=False) \
    .head(10)


,avg_rating,hospital_count
County/Parish,,
LANCASTER,3.7500,4
BUCKS,3.6000,5
ALLEGHENY,3.4375,16
PHILADELPHIA,3.2000,10
MONTGOMERY,2.8750,8
LUZERNE,2.7500,4


In [8]:
#Insights into emergency services

pa.groupby("Emergency Services")["Hospital overall rating"].mean()

Emergency Services
No     4.333333
Yes    3.252101
Name: Hospital overall rating, dtype: float64

In [9]:
pa["Emergency Services"].value_counts()

# No meaningful difference here, just noise

Emergency Services
Yes    119
No       3
Name: count, dtype: int64

In [10]:
#Looking into ownership differences

ownership_summary= (
    pa.groupby("Hospital Ownership")
    .agg(
        avg_rating=("Hospital overall rating", "mean"),
        hospital_count=("Facility ID", "count")
    )
    .sort_values("avg_rating", ascending=False)
)

ownership_summary

,avg_rating,hospital_count
Hospital Ownership,,
Veterans Health Administration,4.400000,5
Voluntary non-profit - Private,3.285714,98
Voluntary non-profit - Other,3.250000,12
Proprietary,2.500000,6
Voluntary non-profit - Church,2.000000,1


In [11]:
#Remove small samples

ownership_summary[ownership_summary["hospital_count"] >=5] \
    .sort_values("avg_rating", ascending=False)

,avg_rating,hospital_count
Hospital Ownership,,
Veterans Health Administration,4.400000,5
Voluntary non-profit - Private,3.285714,98
Voluntary non-profit - Other,3.250000,12
Proprietary,2.500000,6


In [12]:
#Prints cleaned data to .csv

pa.to_csv("../data_clean/pa_hospitals_clean.csv", index=False)
print("Saved: data_clean/pa_hospitals_clean.csv")

Saved: data_clean/pa_hospitals_clean.csv
